# Lab 20: Decomposing Economic Time Series
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 68 min Core + 15 min Write it yourself + 15 min Bonus

---

**Learning Objectives:**
- Decompose an economic time series into trend, seasonal, and residual components using classical and STL methods
- Apply ADF and KPSS stationarity tests and interpret results using the 2×2 decision table
- Detect structural breaks in macroeconomic data with the PELT algorithm
- Distinguish multiplicative from additive seasonality and choose the appropriate decomposition
- Finish a function that rebuilds classical decomposition's seasonal factors with pandas

**Dataset:** FRED economic time series (Retail Sales, Real GDP, Boston Case–Shiller), pulled live
with your FRED key. FRED adds new months and revises old ones, so your counts and last digits can
differ a little from the expected outputs quoted below (measured on 4 October 2026).

**Prerequisites:** Lab 2 (your FRED key, saved as the Colab secret `FRED_API_KEY`), and Lab 1's `.groupby()`.

**How this lab works:** Parts 1-4 are GUIDED: run each cell as it is, then answer the questions.
Parts 5 and 6 each have short blanks (`____`) for you to fill in. "Write it yourself" is required:
you finish a short function that the cell starts for you. Part 7 is an optional bonus.

**Plan on about 1¾ hours** (1 hour 45 minutes) for the required work: setup (about 8 min),
Parts 1-6 (68), Write it yourself (15), and the README and submission (12).

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 5, Step 5:** one blank (`____`): the pandas method that turns GDP into its change from the previous quarter.
> - **Part 6, Step 6:** two blanks (`____`), one word or number each: PELT's cost model and its penalty. Question 2 then has you rerun the cell at two other penalties.
> - **Write it yourself (required),** after Part 6: finish `seasonal_index()`, a function that rebuilds Step 2's twelve seasonal factors with pandas. The cell gives you the trend; you write about three lines: the ratio, the mean for each calendar month, and the rescaling. The check cell below it must print "Passed"; it then uses your function to compare December in two decades.
> - The answers to the interpretation questions, in text cells: two each for Parts 1, 3, 5 and 6, and one for Part 2.
>
> **You run and read:** everything else. Setup and Parts 1-4 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The CHECK cell after Write it yourself is different: you only run it and read its message; you do not need to follow its code.
>
> **Part 7** (optional bonus) is run-and-read, with three questions. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

`fredapi` pulls series from FRED, `statsmodels` has the decompositions and the two stationarity
tests, and `ruptures` has PELT, the break detector in Part 6. Colab lacks `fredapi` and `ruptures`,
so Step 0 installs them (`-q` keeps the install log short).

Step 0's last cell reads your FRED key from Colab Secrets, as in Lab 2. If Colab asks whether this
notebook may read `FRED_API_KEY`, allow it.

In [ ]:
# GUIDED — run as-is
# Step 0: install the packages this lab uses
!pip install fredapi statsmodels ruptures -q

In [ ]:
# GUIDED — run as-is
# Step 0, continued: imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")     # hides library notices; Part 4 explains the one that matters

from fredapi import Fred
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller, kpss
import ruptures as rpt                # PELT, for Part 6

In [ ]:
# GUIDED — run as-is
# Step 0, continued: your FRED key, from Colab Secrets
# Your FRED key comes from Colab's secret store, never from this cell:
#   left sidebar -> key icon (Secrets) -> + Add new secret
#   Name: FRED_API_KEY    Value: your 32-character key    Notebook access: ON
# Free key: https://fredaccount.stlouisfed.org/apikeys
# You set this ONCE; every notebook on the same Google account then finds it.
FRED_API_KEY = ""
# Try the Colab secret; if that fails (not in Colab, or no secret, or no notebook
# access), fall back to an environment variable, then to a hidden prompt.
try:
    from google.colab import userdata
    FRED_API_KEY = userdata.get("FRED_API_KEY") or ""
except Exception:
    import os
    FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

if not FRED_API_KEY:
    # No secret set. getpass hides what you type, so the key stays out of this
    # notebook and out of its saved output either way.
    import getpass
    FRED_API_KEY = getpass.getpass("FRED API key (hidden as you type): ").strip()

if len(FRED_API_KEY) < 20:
    raise ValueError(
        "No FRED API key found. Get one free at "
        "https://fredaccount.stlouisfed.org/apikeys, then add it as a Colab "
        "secret named FRED_API_KEY with Notebook access switched ON -- missing "
        "that switch is the most common cause of this message."
    )

fred = Fred(api_key=FRED_API_KEY)

print("Libraries loaded. Ready to pull FRED data.")

## Part 1: Retail Sales — Visual Inspection (GUIDED — 8 min)

Start by plotting the series, and look for three things: a trend, a seasonal pattern, and
irregular shocks. The plot tells you which decomposition to use.

Step 1 pulls `RSXFSN`, U.S. retail sales **not** seasonally adjusted, because we want to see the
seasonal pattern. FRED returns a Series dated by month. `pd.DatetimeIndex(...)` makes sure the
dates are real dates, and `.asfreq("MS")` puts the series on a regular grid of month starts ("MS"),
which the decompositions in Parts 2 and 3 need. `ax.axvspan(...)` shades March to June 2020.

In [ ]:
# GUIDED — run as-is
# Step 1: pull RSXFSN (retail sales, not seasonally adjusted) from 2000, and plot it
retail = fred.get_series("RSXFSN", observation_start="2000-01-01")
retail = retail.dropna()
retail.index = pd.DatetimeIndex(retail.index)
retail = retail.asfreq("MS")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(retail, color="navy", linewidth=0.8)
ax.axvspan("2020-03", "2020-06", alpha=0.15, color="red", label="COVID-19")
ax.set_title("U.S. retail sales, not seasonally adjusted")
ax.set_ylabel("millions of dollars")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Date range: {retail.index[0].date()} to {retail.index[-1].date()}")
print(f"Observations: {len(retail)}")

**Expected output:** a chart, then `Date range: 2000-01-01 to 2026-08-01` and `Observations: 320`
(one more for each month FRED has added since 4 October 2026).

### Interpretation Questions

1. Does the seasonal zig-zag get **larger** as the trend rises? From that, would you choose an additive or a multiplicative decomposition, and why?
2. Can you spot the COVID-19 shock? How does it differ from the normal seasonal dips?

*Your answers here:*

1.
2.

## Part 2: Classical Decomposition (GUIDED — 12 min)

Classical decomposition splits $Y_t = T_t \times S_t \times R_t$ (multiplicative) in four moves:

1. **Trend** $T_t$: a centred 12-month moving average, so each month's trend averages one full year around it.
2. **Ratio** $Y_t / T_t$: what is left once the trend is divided out.
3. **Seasonal factors** $S_t$: the average ratio for each calendar month (all the Januaries, all the Februaries, ...), rescaled so the twelve average 1. Every January gets the same factor.
4. **Residual** $R_t = Y_t / (T_t \times S_t)$: whatever is left.

It is the simplest approach, and it has real limitations we will see in Part 3. In "Write it
yourself" you finish a function that rebuilds moves 1-3; move 1, the trend, is given.

Steps 2 and 3 both draw four panels, so the first cell defines a small helper, `plot_parts`.
`zip(axes, parts, names)` walks through the three lists together, one panel at a time. The dashed
line in the residual panel marks "nothing left over": 1 for a ratio, 0 for a difference.

In [ ]:
# GUIDED — run as-is
# Step 2: a helper that draws the four panels of a decomposition
def plot_parts(result, title, baseline):
    """Plot the observed series, trend, seasonal part and residual, one under another."""
    parts = [result.observed, result.trend, result.seasonal, result.resid]
    names = ["observed", "trend", "seasonal", "residual"]
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    for ax, part, name in zip(axes, parts, names):
        ax.plot(part, color="navy", linewidth=0.8)
        ax.set_ylabel(name)
    axes[3].axhline(baseline, color="gray", linestyle="--", linewidth=0.5)
    axes[0].set_title(title)
    plt.tight_layout()
    plt.show()

`seasonal_decompose` does the four moves. `period=12` because retail sales are monthly with an
annual cycle. The seasonal component repeats every year, so its first 12 values (January to
December 2000, since the series starts in January) are all twelve factors. `.iloc[:12]` takes the
first 12 rows by position.

In [ ]:
# GUIDED — run as-is
# Step 2, continued: classical multiplicative decomposition of retail sales
classical = seasonal_decompose(retail, model="multiplicative", period=12)
plot_parts(classical, "Classical multiplicative decomposition (retail sales)", baseline=1)

print("Seasonal factors, January to December:")
print(classical.seasonal.iloc[:12].round(4).values)

**Expected output:** four panels, then twelve factors:
`[0.9032 0.8886 1.0103 0.9863 1.0454 1.0126 1.0145 1.0342 0.9606 0.9909 1.0068 1.1466]`.
With a month or two more data your last digits may move by a few ten-thousandths.

### Interpretation Questions

1. Which month has the highest seasonal factor, and how far above trend does it put that month? What economic explanation can you give?

*Your answers here:*

1.

## Part 3: STL Decomposition (GUIDED — 12 min)

STL (Seasonal-Trend decomposition using LOESS) lets the seasonal component change from year to
year, where classical decomposition gives every January the same factor. `robust=True` changes how
STL treats outliers.

STL is additive, so for multiplicative data we take logs first:
$\log(Y_t) = \log(T_t) + \log(S_t) + \log(R_t)$.

`STL(...)` sets up the model and `.fit()` runs it; the result has the same four parts, so
`plot_parts` draws it. The last two lines compare the residuals' spread. A ratio of 1.02 and a log
of 0.02 both mean "2% above", so the two standard deviations are on about the same scale.

In [ ]:
# GUIDED — run as-is
# Step 3: STL on the log of retail sales, robust to outliers
log_retail = np.log(retail)
stl_result = STL(log_retail, period=12, robust=True).fit()
plot_parts(stl_result, "STL decomposition (log retail sales, robust=True)", baseline=0)

classical_resid_std = classical.resid.dropna().std()     # the first and last six months have no trend
stl_resid_std = stl_result.resid.std()
print(f"Classical residual std: {classical_resid_std:.4f}")
print(f"STL residual std (log scale): {stl_resid_std:.4f}")

**Expected output:** four panels, then `Classical residual std: 0.0205` and
`STL residual std (log scale): 0.0238`.

### Interpretation Questions

1. Compare the **seasonal panel** in STL with the classical one. Does the STL seasonal component change over time? Where do you see it shifting?
2. Look at the **residual panels** of Steps 2 and 3 around 2020, and at the two printed standard deviations. What does the spike tell you about how each method handles a one-time shock? Is it smaller or larger in STL, and what does `robust=True` do to the trend, and so to the residual?

*Your answers here:*

1.
2.

## Part 4: Stationarity Testing — ADF & KPSS (GUIDED — 12 min)

Most time series models (ARIMA, VAR) require **stationary** data — constant
mean and variance over time. We use two tests with **opposite null hypotheses**
to reach a confident conclusion:

| | ADF rejects (p < 0.05) | ADF fails to reject |
|---|---|---|
| **KPSS rejects** | **Conflict** — often a trend or a break; rerun both with `regression="ct"` before deciding | **Non-stationary** — difference it |
| **KPSS fails to reject** | **Stationary** — model it as it is | **Inconclusive** — more data, or another test |

Step 4 pulls real GDP (`GDPC1`, quarterly) from 1990, the chapter's sample. `"QS"` is a grid of
quarter starts.

In [ ]:
# GUIDED — run as-is
# Step 4: pull GDPC1 (real GDP, quarterly) from 1990, and plot it
gdp = fred.get_series("GDPC1", observation_start="1990-01-01")
gdp = gdp.dropna()
gdp.index = pd.DatetimeIndex(gdp.index)
gdp = gdp.asfreq("QS")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(gdp, color="navy", linewidth=1.0)
ax.set_title("U.S. real GDP (GDPC1, quarterly)")
ax.set_ylabel("billions of chained 2017 dollars")
plt.tight_layout()
plt.show()

print(f"Quarters: {len(gdp)}, {gdp.index[0].date()} to {gdp.index[-1].date()}")

**Expected output:** a chart, then `Quarters: 146, 1990-01-01 to 2026-04-01` (147 once FRED adds
2026 Q3).

Part 5 runs the same two tests again, so Step 4 wraps them in a helper, `two_tests`.

- `adfuller` gives back six results; the first two are the test statistic and the p-value, and the
  fifth is a dictionary of critical values. `kpss` gives back four: statistic, p-value, lags used,
  and critical values. A name of just `_` is the usual way to say "a result we do not need".
  `autolag="AIC"` and `nlags="auto"` let each test choose how many lags to use, and
  `regression="c"` makes KPSS's null "stationary around a constant mean".
- `crit["5%"]` looks up the 5% critical value in that dictionary.
- The `if` / `elif` chain is the 2×2 table above, at the 5% level.

KPSS looks its p-value up in a short table that runs from 0.01 to 0.10. A p-value printed as
`0.0100` means "0.01 or less", and `0.1000` means "0.10 or more". (statsmodels says so in a warning,
which Step 0 hid.)

In [ ]:
# GUIDED — run as-is
# Step 4, continued: ADF and KPSS on GDP in levels, and the 2x2 table's conclusion
def two_tests(series):
    """Print ADF and KPSS for one series, and the 2x2 table's conclusion at the 5% level."""
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series, autolag="AIC")
    kpss_stat, kpss_p, _, kpss_crit = kpss(series, regression="c", nlags="auto")

    print(f"{'test':<6}{'H0':<12}{'statistic':>11}{'p-value':>10}{'5% critical':>13}")
    print(f"{'ADF':<6}{'unit root':<12}{adf_stat:>11.4f}{adf_p:>10.4f}{adf_crit['5%']:>13.4f}")
    print(f"{'KPSS':<6}{'stationary':<12}{kpss_stat:>11.4f}{kpss_p:>10.4f}{kpss_crit['5%']:>13.4f}")

    if adf_p < 0.05 and kpss_p >= 0.05:
        conclusion = "stationary: model it as it is"
    elif adf_p >= 0.05 and kpss_p < 0.05:
        conclusion = "non-stationary: difference it"
    elif adf_p < 0.05 and kpss_p < 0.05:
        conclusion = 'conflict: often a trend or a break; rerun both with regression="ct"'
    else:
        conclusion = "inconclusive: more data, or another test"
    print(f"2x2 table at the 5% level: {conclusion}")

two_tests(gdp)

**Expected output:** ADF statistic `0.9690`, p-value `0.9939`; KPSS statistic `1.6916`, p-value
`0.0100`; and `2x2 table at the 5% level: non-stationary: difference it`. FRED revises GDP, so the
statistics can move in the second decimal; the conclusion should not.

## Part 5: First-Differencing to Achieve Stationarity (YOUR TASK — 10 min)

GDP has a unit root: its level wanders without returning to a fixed mean. The standard fix is
**first-differencing**: $\Delta Y_t = Y_t - Y_{t-1}$. For GDP, that is the change in real GDP from
one quarter to the next, in billions of dollars (not a percentage growth rate).

Step 5 is yours: one blank. The first quarter has no quarter before it, so its difference is NaN,
and `.dropna()` removes it. The chart puts the level beside the difference.

In [ ]:
# YOUR TASK — fill in the blank (____)
# Step 5: first-difference GDP, and plot the level beside the difference
gdp_diff = gdp.____().dropna()       # which pandas Series method gives Y_t - Y_(t-1)?

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(gdp, color="navy")
axes[0].set_title("GDP level")
axes[0].set_ylabel("billions of dollars")
axes[1].plot(gdp_diff, color="seagreen")
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.5)
axes[1].set_title("GDP first difference")
axes[1].set_ylabel("change in billions of dollars")
plt.tight_layout()
plt.show()

Step 5, continued, runs Step 4's two tests on the differenced series.

In [ ]:
# GUIDED — run as-is
# Step 5, continued: ADF and KPSS on the first difference
two_tests(gdp_diff)

**Expected output:** ADF statistic `-14.2297`, p-value `0.0000` (far below 0.0001); KPSS statistic
`0.2050`, p-value `0.1000` (that is, 0.10 or more); and
`2x2 table at the 5% level: stationary: model it as it is`. An ADF statistic near `-13.69` with a
KPSS statistic near `0.127` means you took percentage changes (`pct_change`), which is Part 6's
series, not the difference.

### Interpretation Questions

1. How does the visual character of the series change after differencing? Does it look mean-reverting?
2. What does the large negative spike in 2020 Q2 represent in economic terms?
3. If the ADF p-value after differencing is very small, GDP is said to be I(1) — "integrated of order 1." What would I(2) mean?

*Your answers here:*

1.
2.
3.

## Part 6: Structural Break Detection with PELT (YOUR TASK — 14 min)

The 2×2 stationarity table assumes the data-generating process doesn't change. But economies **do**
change: recessions, policy shifts, pandemics. The PELT (Pruned Exact Linear Time) algorithm finds
**changepoints** where the statistical properties of the series shift.

Step 6 runs PELT on quarterly GDP growth in percent: `.pct_change()` is the change from the previous
quarter as a share of it, so `* 100` makes it a percent. `.values` hands ruptures a plain array of
the numbers.

- `rpt.Pelt(model=...)` sets the **cost model**, a string naming what counts as a change. Lecture
  used the one that detects a change in the whole distribution, not only its mean.
- `.predict(pen=...)` sets the **penalty** each extra break must pay. Higher means fewer breaks. The
  right penalty depends on the cost model and on how many points there are; with this cost model,
  sensible penalties are small numbers. Start from the penalty on the lecture's PELT slide.

`predict` gives back the position where each segment ends, and the last one is always the length of
the series. The list comprehension `[... for bp in breakpoints if bp < len(gdp_growth)]` drops that
last one and turns each other position into a date: `gdp_growth.index[bp - 1]` is the **last
quarter before** each change.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 6: detect structural breaks in GDP growth with PELT
gdp_growth = gdp.pct_change().dropna() * 100      # quarterly growth, in percent

signal = gdp_growth.values
algo = rpt.Pelt(model=____).fit(signal)           # blank 1: the cost model, a string
breakpoints = algo.predict(pen=____)              # blank 2: the penalty, a number
break_dates = [gdp_growth.index[bp - 1] for bp in breakpoints if bp < len(gdp_growth)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_growth, color="navy", linewidth=0.8)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
for bd in break_dates:
    ax.axvline(bd, color="firebrick", linestyle="--", alpha=0.7)
ax.set_title("GDP growth with PELT structural breaks")
ax.set_ylabel("quarterly growth (%)")
plt.tight_layout()
plt.show()

print(f"Number of detected breakpoints: {len(break_dates)}")
print(f"Break dates: {[str(d.date()) for d in break_dates]}")

**Expected output:** `Number of detected breakpoints: 4`, at `1991-04-01`, `2000-01-01`,
`2020-01-01` and `2021-04-01`. If you see 0 breaks, your penalty is too high for this series. Four
breaks at 1991-04, 2000-01, **2007-07** and **2010-01** mean the cost model looks only for shifts in
the mean (`"l2"`), not the whole distribution. 21 breaks means `"normal"`. `"l1"` happens to hit
these same four dates, so check your string against the slide.

### Interpretation Questions

1. Do the detected breakpoints correspond to known economic events (recessions, policy changes, pandemics)?
2. Rerun Step 6 with `pen=0.5` and then `pen=2`, and write down how many breaks each finds; then set it back to the original penalty and run it once more. What is the penalty trading off?

*Your answers here:*

1.
2.

---

## Write it yourself (required — 15 min)

This part is required. The check cell below it must print "Passed" before you submit.

**The task.** Step 2's `seasonal_decompose` printed twelve seasonal factors for retail sales. Finish
a function that rebuilds them with pandas, for any monthly series.

The cell below starts `seasonal_index(series)` for you. It sorts the series by date and builds Part
2's move 1, the trend. Twelve months have no middle month, so the trend at month *t* is the mean of
two 12-month averages: months *t*−6 to *t*+5, and months *t*−5 to *t*+6.
`y.rolling(12, center=True).mean()` gives the first, at every month; `.shift(-1)` moves each value
up one row, so at month *t* it holds the next month's average. The first and last six trend values
are NaN (not a number): no full window fits there. Leave those lines as they are.

Add moves 2 and 3, and give back the twelve factors, January first:

2. **Ratio:** the series divided by its trend.
3. **Factors:** the mean ratio for each calendar month (all the Januaries, all the Februaries, ...),
   rescaled so the twelve average 1.

- `y.index.month` gives each date's month number, 1 to 12, and `.groupby()` (Lab 1) can group by
  it. `.mean()` skips the NaN ratios at the two ends.
- Use the function's own `y` and `trend`: the check also runs it on another stretch of retail sales.
- Give back the full numbers; round only when you print.

It takes about three lines. Then run the check cell. It compares your factors with Step 2's and with
`seasonal_decompose` on July 2005 to December 2019. `assert` stops with its message when its
condition is False; the message tells you what to fix.

In [ ]:
# WRITE IT YOURSELF (required): seasonal_index(series)
def seasonal_index(series):
    """Classical multiplicative seasonal factors of a monthly series, January first."""
    y = series.sort_index()
    early = y.rolling(12, center=True).mean()      # at month t: the average of months t-6 to t+5
    trend = (early + early.shift(-1)) / 2          # averaged with months t-5 to t+6, so centred on t
    # Your lines below: the ratio, the mean ratio for each calendar month, the rescaling, and return

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
written = "seasonal_index" in globals()
if written:
    # try/except: an error here is reported, with its cause, further down
    try:
        written = seasonal_index(retail.copy()) is not None
    except Exception:
        pass
if not written:
    print("Not written yet: finish seasonal_index() in the cell above (the ratio, the mean for each "
          "calendar month, the rescaling and a return line), run it, then run this check.")
else:
    assert len(classical.observed) == len(retail) and np.allclose(classical.observed, retail), (
        "retail or classical has changed since Steps 1-2. Write only inside the function above, "
        "rerun Steps 1-2, then this cell.")

    # the names your function reads from outside itself
    outside = set(seasonal_index.__code__.co_names)
    assert not outside & {"seasonal_decompose", "STL"}, (
        "seasonal_index calls the library. Rebuild the factors with pandas, as the task says.")
    lab_names = sorted(outside & {"retail", "log_retail", "classical", "stl_result", "gdp", "gdp_diff"})
    assert not lab_names, (
        f"seasonal_index uses {', '.join(lab_names)} from the lab inside the function. Use only its "
        "argument, so it works on any monthly series.")

    def run_it(series):
        """Call your function on a copy of series, and hand back its twelve factors, January first."""
        # try/except catches an error inside your function, so the message can name the cause
        try:
            out = seasonal_index(series.copy())
        except Exception as e:
            hint = "Read the error above: it says what went wrong."
            if "positional argument" in str(e):
                hint = "seasonal_index takes one argument: the monthly series."
            elif isinstance(e, KeyError):
                hint = ("Your function looks something up by name. It is handed a Series, not a table, "
                        "and its name can be anything: use the series itself.")
            elif "month" in str(e):
                hint = ("Your function lost the dates. Keep the series' date index: the month of each "
                        "value comes from it.")
            # str(e)[:200] keeps the first 200 characters of the error message
            raise AssertionError(f"seasonal_index stopped with an error:\n  {str(e)[:200]}\n{hint}") from None
        assert out is not None, "seasonal_index gave back nothing. Does it end with a return line?"
        if isinstance(out, pd.Series) and out.index.inferred_type in ("string", "mixed"):
            raise AssertionError(
                "Your factors are labelled by month names, and month names sort alphabetically (April "
                "first). Group by y.index.month, the month number, so January comes first.")
        try:
            values = np.ravel(np.asarray(out, dtype=float))
        except (TypeError, ValueError):
            raise AssertionError(
                f"seasonal_index gives back a {type(out).__name__}, not numbers. Give back the twelve "
                "factors as a pandas Series (or a list of numbers), January first.") from None
        assert len(values) != len(series), (
            f"seasonal_index gives back {len(values)} values, one per month of the series. Average the "
            "ratios by calendar month, to get twelve factors.")
        assert len(values) != 4, (
            "seasonal_index gives back 4 values, one per quarter. Group by month (1-12), to get twelve "
            "factors.")
        assert len(values) == 12, (
            f"seasonal_index gives back {len(values)} value(s). Give back twelve: one factor per "
            "calendar month, January first.")
        if isinstance(out, pd.Series) and sorted(out.index) == list(range(1, 13)):
            values = out.sort_index().to_numpy(dtype=float)     # a Series labelled by month 1-12
        assert not np.isnan(values).any(), (
            "Some of your factors are NaN. The first and last six trend values are NaN; .mean() skips "
            "them, but other ways of averaging do not.")
        return values

    def same_months(got, target, where):
        """Stop if got is target with every factor moved to another month."""
        for k in range(1, 12):
            assert not np.allclose(got, np.roll(target, k), rtol=1e-6), (
                f"On {where}, your factors land in the wrong months: each sits {k} month(s) away from "
                "its own. Group by each date's calendar month, not by position in the series.")

    def library(series):
        """seasonal_decompose's twelve factors for series, January first."""
        first12 = seasonal_decompose(series, model="multiplicative", period=12).seasonal.iloc[:12]
        return np.roll(first12.to_numpy(), series.index[0].month - 1)

    # 1. retail sales, against the factors Step 2 printed (retail starts in January)
    want = classical.seasonal.iloc[:12].to_numpy()
    mine = run_it(retail)
    for places in range(1, 9):
        assert not (np.allclose(mine, want.round(places), rtol=0, atol=1e-12)
                    and not np.allclose(mine, want, rtol=1e-9, atol=0)), (
            "Your factors are rounded. Give back the full numbers; round only when you print.")
    scale = mine / want
    assert not (np.allclose(scale, scale[0], rtol=1e-6) and not np.isclose(scale[0], 1)), (
        f"Your factors have the right pattern, but they average {mine.mean():.4f}, not 1. Rescale "
        "them: divide each by the mean of the twelve.")
    same_months(mine, want, "retail sales")
    assert (mine > 0.2).all() and (mine < 5).all(), (
        "Your factors are not ratios near 1. Did you subtract the trend? In a multiplicative "
        "decomposition the series is divided by its trend.")
    upside_down = (1 / want) / (1 / want).mean()
    assert not (np.allclose(mine, upside_down, rtol=5e-3) and not np.allclose(mine, want, rtol=5e-3)), (
        "Your factors are upside down: December comes out lowest. Divide the series by its trend, "
        "not the trend by the series.")

    # 2. another stretch of retail sales, starting in July, and the same months shuffled
    part = retail["2005-07":"2019-12"].rename("sales")
    want2 = library(part)
    mine2 = run_it(part)
    assert not np.allclose(mine2, mine, rtol=1e-9), (
        "seasonal_index gives the same factors for 2005-2019 as for the whole series. Does it use its "
        "argument everywhere?")
    same_months(mine2, want2, "retail sales from July 2005")
    if np.allclose(mine2, want2, rtol=1e-9, atol=0):
        shuffled = part.sample(frac=1, random_state=0)     # the same months in a random order
        assert np.allclose(run_it(shuffled), want2, rtol=1e-9, atol=0), (
            "Your function is right when the months arrive in date order, and wrong when they do not. "
            "Sort the series by date first.")

    # 3. two made-up monthly series whose factors are known exactly
    months = pd.date_range("2001-01-01", periods=144, freq="MS")
    pattern = np.array([0.90, 0.89, 1.01, 0.99, 1.05, 1.01, 1.01, 1.03, 0.96, 0.99, 1.01, 1.15])
    flat = pd.Series(1000 * np.tile(pattern, 12), index=months)    # no growth, the same 12 months every year
    same_months(run_it(flat), pattern / pattern.mean(), "a made-up series")
    assert np.allclose(run_it(flat), pattern / pattern.mean(), rtol=1e-9, atol=0), (
        "Your trend window is not exactly one year. On a series that repeats the same 12 months with no "
        "growth, every 12-month window has the same mean, so the factors must equal the pattern "
        "(rescaled to average 1). An 11- or 13-month window leaves a month out or counts one twice; or "
        "the first and last six trend values are not NaN (min_periods, fillna, bfill): leave them NaN. "
        "Did you change the given trend lines? Put them back.")
    line = pd.Series(10.0 + np.arange(144), index=months)          # a straight line, no seasonal pattern
    off = np.abs(run_it(line) - 1).max()
    if off > 6e-3:
        cause = ("Your trend looks back, not both ways: a window that ends at month t lags the series "
                 "by about six months.")
    elif off > 1.8e-3:
        cause = ("Your trend is centred on a neighbouring month, not on month t: a window or a .shift() "
                 "points the wrong way (shift(-1) brings month t+1's value to month t).")
    else:
        cause = ("Your trend is half a month off centre. Twelve months have no middle month, so one "
                 "12-month window cannot be centred on month t; the task averages two of them.")
    assert off < 1e-9, (
        f"{cause} On a straight line with no seasonal pattern, every factor should be exactly 1; yours "
        f"are off by up to {off:.4f}. The trend at month t is the mean of two 12-month averages: months "
        "t-6 to t+5, and months t-5 to t+6. Or the first and last six trend values are not NaN "
        "(min_periods, fillna, bfill): leave them NaN. Did you change the given trend lines? Put them back.")

    # 4. the exact numbers: Step 2's twelve, and seasonal_decompose's on the July 2005 stretch
    for got, target, where in [(mine, want, "retail sales (Step 2)"),
                               (mine2, want2, "retail sales, July 2005 to December 2019")]:
        worst = int(np.argmax(np.abs(got - target)))
        assert np.allclose(got, target, rtol=1e-9, atol=0), (
            f"On {where}, your factor for month {worst + 1} is {got[worst]:.6f}; seasonal_decompose "
            f"gives {target[worst]:.6f}. Your trend passes the tests above, so look at the averaging: "
            "take the mean of each calendar month's ratios, keeping every month (a median, or dropping "
            "some months, gives slightly different factors).")

    december_early = run_it(retail["2000-01":"2009-12"])[11]
    december_late = run_it(retail["2015-01":"2024-12"])[11]
    print("Passed. seasonal_index matches Step 2's twelve factors, and seasonal_decompose on retail "
          "sales from July 2005 to December 2019, in any order.")
    print(f"December's factor, 2000-2009: {december_early:.4f};  2015-2024: {december_late:.4f}")

**Expected output:** `Passed. ...`, then December's factor for 2000-2009 and for 2015-2024 (measured
on 4 October 2026: `1.1637` and `1.1274`).

---

## Part 7: BONUS — Boston Case-Shiller Decomposition (Optional — 15 min)

The same tools, on a different series: the Boston Case–Shiller Home Price Index, not seasonally
adjusted, which changes sharply after 2020.

Step 7 pulls `BOXRHTNSA`, the raw index. `BOXRHTSA` is the seasonally adjusted one: STL on that
series measures a seasonal pattern that has already been removed. Its seasonal panel still moves,
but swings only about 3.7% from peak to trough, against about 6.2% on the raw index, so you would be
decomposing the adjuster's leftovers, not the housing market.

In [ ]:
# GUIDED — run as-is (bonus)
# Step 7: pull BOXRHTNSA (Boston home prices, not seasonally adjusted), and plot it
boston_hpi = fred.get_series("BOXRHTNSA", observation_start="2000-01-01")
boston_hpi = boston_hpi.dropna()
boston_hpi.index = pd.DatetimeIndex(boston_hpi.index)
boston_hpi = boston_hpi.asfreq("MS")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(boston_hpi, color="navy", linewidth=0.8)
ax.axvspan("2020-03", "2020-06", alpha=0.15, color="red", label="COVID-19")
ax.set_title("Boston Case-Shiller home price index (not seasonally adjusted)")
ax.set_ylabel("index value")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# GUIDED — run as-is (bonus)
# Step 7, continued: STL on the log of the index, as in Step 3
log_boston = np.log(boston_hpi)
stl_boston = STL(log_boston, period=12, robust=True).fit()
plot_parts(stl_boston, "STL decomposition (log Boston Case-Shiller)", baseline=0)

Step 7 uses `pen=2`; at `pen=10` PELT finds no breaks in this series.

In [ ]:
# GUIDED — run as-is (bonus)
# Step 7, continued: PELT on monthly growth in the index
boston_growth = boston_hpi.pct_change().dropna() * 100
algo_b = rpt.Pelt(model="rbf").fit(boston_growth.values)
breaks_b = algo_b.predict(pen=2)
break_dates_b = [boston_growth.index[bp - 1] for bp in breaks_b if bp < len(boston_growth)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(boston_growth, color="navy", linewidth=0.8)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
for bd in break_dates_b:
    ax.axvline(bd, color="firebrick", linestyle="--", alpha=0.7)
ax.set_title("Boston home price growth with PELT structural breaks")
ax.set_ylabel("monthly growth (%)")
plt.tight_layout()
plt.show()

print(f"Break dates: {[str(d.date()) for d in break_dates_b]}")

**Expected output:** seven break dates: `2001-09-01`, `2002-02-01`, `2005-06-01`, `2012-12-01`,
`2020-01-01`, `2022-07-01`, `2022-12-01`.

### Bonus Interpretation

1. Does the STL trend show a visible acceleration after 2020? What drove Boston home prices up so dramatically?
2. Compare the 2008 financial crisis and the 2020 pandemic in the residual panel. Which was more disruptive to the trend?
3. Do the PELT breakpoints line up with the 2008 crisis and the 2020 pandemic? Are there any breaks you did not expect?

*Your answers here:*

1.
2.
3.

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Pulled FRED series with fredapi: U.S. retail sales, not seasonally
  adjusted (RSXFSN, [YOUR VALUE] months), and real GDP (GDPC1, from 1990)
* Ran a classical multiplicative decomposition of retail sales: the
  December seasonal factor was [YOUR VALUE], and the lowest month was
  [YOUR MONTH] at [YOUR VALUE]
* Ran STL on log retail sales with outliers downweighted: residual
  standard deviation [YOUR VALUE], against [YOUR VALUE] for classical
* Tested GDP for stationarity with ADF and KPSS: the 2x2 table said
  [YOUR RESULT, LEVELS] in levels and [YOUR RESULT, DIFFERENCE]
  after first-differencing
* Found structural breaks in quarterly GDP growth with PELT (ruptures,
  [YOUR COST MODEL] cost, penalty [YOUR VALUE]): [YOUR DATES]
* Finished a function, seasonal_index(), that rebuilds the classical
  seasonal factors with pandas; December's factor was [YOUR VALUE] in
  2000-2009 and [YOUR VALUE] in 2015-2024
* (Only if I did the bonus) Boston home prices: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Decomposing Economic Time Series
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive',
'leverage' or 'cutting-edge'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab20-time-series`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab20-time-series`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab20-time-series`, branch `main`,
   commit message `Lab 20 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.